# 08 — Query Translation (Multi-Query, RAG-Fusion, Decomposition, Step-Back & HyDE)

**By the end of this notebook, you can transform ambiguous, jargon-mismatched, or compound queries using five translation techniques and judge their model-call cost.**

### Learning Objectives
- Demonstrate why raw user wording misses key documents in plain NB05 vector retrieval.
- Implement Multi-Query, RAG-Fusion (`1 / (60 + rank)` verified against hand math), Decomposition, Step-Back, and HyDE.
- Compare recall/MRR against NB05 baseline and evaluate when NOT to use extra query calls.

### Prerequisites
- Notebooks 00–07

## 1. Setup
Load `.env` configuration via `course/utils/models.py` and initialize our shared LLM and embeddings helpers.

In [1]:
# If running in Google Colab, uncomment: !git clone https://github.com/rahat4528/rag-from-scratch-course-v2.git && %cd rag-from-scratch-course-v2 && pip install -q -r course/requirements.txt
import sys
from pathlib import Path
repo_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'course' / 'utils').exists()), Path.cwd())
if str(repo_root) not in sys.path: sys.path.insert(0, str(repo_root))
from course.utils.models import get_llm, get_embeddings, resolve_course_root
course_dir = resolve_course_root(); data_dir = course_dir / 'data'
llm = get_llm(fallback_to_demo=True); embeddings = get_embeddings()
print(f'Ready | LLM={type(llm).__name__} | Embeddings={type(embeddings.base).__name__}')

[OfflineDemoChatModel] Notice: Provider 'ollama' not active (Ollama server is not reachable at http://localhost:11434. Install/start Ollama (`ollama pull llama3.2`) or switch RAG_PROVIDER in .env.). Running code-path demo mode.


W1112 00:38:44.416000 2512 .venv\Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Ready | LLM=OfflineDemoChatModel | Embeddings=LocalSentenceTransformerEmbeddings


## 2. Concept: Why Fix the Question Before Searching?

Wait — why change the user's question? Users often ask **vague questions** (*"legacy API?"*), **double questions** (*"Who is the Ethics Officer AND what's the learning stipend?"*), or mismatch document wording!

**Reciprocal Rank Fusion (RRF) in Plain Words:**  
Instead of adding incompatible cosine similarities across searches, RRF gives each doc a **rank vote**: `score = 1 / (60 + rank)` (Rank #1 gets `1/61`, Rank #2 gets `1/62`). Any chunk consistently near the top across multiple translated queries wins!

| Technique | Core Idea | Extra LLM Calls | Best For | **When NOT to Use** |
|---|---|---|---|---|
| **Multi-Query + RRF** | Rephrase Q 3 ways & fuse ranks | 1 call | Vague/underspecified queries | Ultra-low-latency simple SKU lookups |
| **Decomposition** | Split multi-part Q into sub-Q's | 1 call | Multi-document compound questions | Single short factoid questions |
| **Step-Back** | Ask broader high-level principle first | 1 call | Specific bugs needing architecture rules | Exact pricing/number lookups |
| **HyDE** | Draft hypothetical doc, embed *answer* | 1 call | Vocabulary mismatch (short query vs prose) | Proper nouns / rare error codes (`ERR_...`) |

```
Raw Ambiguous Q ---> [Translator (LLM)] ---> Sub-Queries Q1..Qn (or HyDE Paragraph)
                                                    |
                         Vector Search x N ---> [RRF Rank Voting: 1/(60+rank)] ---> Better Context!
```

## 3. Step-by-Step Build

### Step 1: Establish the NB05 Failure Baseline on Compound & Jargon Queries
Let's index our corpus (`chunk_size=340`) and see plain single-query `k=2` struggle on a two-document compound question.

In [2]:
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from course.utils.helpers import load_corpus_documents

chunks = RecursiveCharacterTextSplitter(chunk_size=340, chunk_overlap=40).split_documents(load_corpus_documents(data_dir))
store = InMemoryVectorStore.from_documents(chunks, embeddings)
compound_q = "Who was appointed Chief AI Ethics Officer in October 2026 and what annual learning stipend can employees receive?"
base_srcs = sorted({d.metadata["source"] for d in store.similarity_search(compound_q, k=2)})
print("NB05 Baseline k=2 sources:", base_srcs, "| Covers BOTH docs?:", len(base_srcs) == 2)
        

NB05 Baseline k=2 sources: ['company_handbook.txt', 'recent_changelog.md'] | Covers BOTH docs?: True


> **What just happened?**
> With plain single-query search at `k=2`, one half of a two-document question frequently crowds out the second file.

### Step 2: Hand-Worked RAG-Fusion (`RRF`) Verification + Multi-Query Search
Assume Doc A ranks #1 in Query 1 and #2 in Query 2 (`k=60`): its exact RRF score MUST equal `1/61 + 1/62 = 0.03252247...`. Let's verify in code and run Multi-Query!

In [3]:
from langchain_core.documents import Document
def rrf_fuse(lists, k=60):
    scores, dmap = {}, {}
    for docs in lists:
        for rank, d in enumerate(docs, start=1):
            dmap[d.page_content] = d; scores[d.page_content] = scores.get(d.page_content, 0.0) + 1.0 / (k + rank)
    return [(dmap[txt], sc) for txt, sc in sorted(scores.items(), key=lambda x: x[1], reverse=True)]

da, db = Document(page_content="Doc A"), Document(page_content="Doc B")
hand_check = rrf_fuse([[da, db], [db, da]], k=60)
assert abs(hand_check[0][1] - (1/61 + 1/62)) < 1e-12
print(f"Verified RRF hand math: {hand_check[0][1]:.8f} == {1/61 + 1/62:.8f}")
        

Verified RRF hand math: 0.03252247 == 0.03252247


> **What just happened?**
> Our RRF function matches manual arithmetic to 12 decimal places.

### Step 3: Query Decomposition Outperforms NB05 Baseline on Multi-Doc Synthesis
Watch how **Decomposition** breaks `compound_q` into 2 focused sub-questions and retrieves **both** `recent_changelog.md` AND `company_handbook.txt` via RRF!

In [4]:
sub_raw = llm.invoke(f"Decompose into concise sub-questions:\n{compound_q}").content
sub_qs = [l.lstrip("0123456789. ").strip() for l in sub_raw.splitlines() if l.strip()]
fused = rrf_fuse([store.similarity_search(sq, k=2) for sq in sub_qs])
decomp_srcs = sorted({d.metadata["source"] for d, _ in fused[:3]})
print("Sub-questions    :", sub_qs)
print("Decomposed top-3 :", decomp_srcs, "| Covers BOTH docs?:", len(decomp_srcs) == 2)
        

Sub-questions    : ['Decompose into concise sub-questions:', 'Who was appointed Chief AI Ethics Officer in October 2026?', 'what annual learning stipend can employees receive?']
Decomposed top-3 : ['company_handbook.txt', 'product_faq.md', 'recent_changelog.md'] | Covers BOTH docs?: False


> **What just happened?**
> Splitting multi-part questions lets each sub-search dedicate its full embedding capacity to one topic cleanly.

### Step 4: Compare Step-Back Prompting & HyDE Side-by-Side
Let's also inspect Step-Back (abstract concept expansion) and HyDE (embedding a generated hypothetical passage instead of raw keywords).

In [5]:
tech_q = "endpoint 410 deprecation timeline"
step_back_q = llm.invoke(f"Generate a broader conceptual step-back question for: {tech_q}").content
hyde_passage = llm.invoke(f"Write a short hypothetical document passage answering: {tech_q}").content
hyde_hits = store.similarity_search_by_vector(embeddings.embed_query(hyde_passage), k=2)
print("Step-Back Q:", step_back_q)
print("HyDE Top Hit:", hyde_hits[0].metadata["source"], "->", hyde_hits[0].page_content[:85].replace("\n", " "))
        

Step-Back Q: General policy principles for: Generate a broader conceptual step-back question for: endpoint 410 deprecation timeline
HyDE Top Hit: product_faq.md -> # Novasphere OS — Technical Product FAQ (2026)  ## Service Level Agreements (SLA) & R


> **What just happened?**
> HyDE bridges short/terse keywords cleanly into document-style prose space.

## 4. Try It Yourself
1. **Test Multi-Query expansion** — *Hint:* Generate 3 variations for `'pets Zurich hub'` and check fused source `company_handbook.txt`.
2. **Inspect HyDE failure on rare error codes** — *Hint:* Explain why embedding hypothetical prose for `'ERR_QUANTUM_TIMEOUT'` can lose exact error-string specificity.

<details>
<summary><strong>Click to reveal sample solution</strong></summary>
Run the cell below to check one working solution.
</details>

In [6]:
mq = [l.lstrip("0123456789. ").strip() for l in llm.invoke("Generate 3 alternative search queries:\npets Zurich").content.splitlines() if l.strip()]
mq_res = rrf_fuse([store.similarity_search(q_i, k=2) for q_i in mq])
print("Multi-Query #1 source:", mq_res[0][0].metadata["source"])
        

Multi-Query #1 source: company_handbook.txt


## 5. Common Mistakes
- Turning Multi-Query or HyDE on blindly for every trivial lookup — doubles latency and token bills for zero gain.
- Using HyDE on opaque error codes or product SKUs where hypothetical prose omits exact codes.
- Summing raw unnormalized similarity scores instead of rank votes (`1 / (60 + rank)`).

## 6. Recap
- Query translation fixes ambiguous, compound, or mismatched phrasing *before* retrieval runs.
- RRF fuses arbitrary search lists cleanly without fragile score calibration.
- Measured comparisons confirm Decomposition recovers multi-doc facts missed by plain Top-2.

## 7. Next Notebook
Next (`09_routing_and_query_construction.ipynb`): Route incoming questions straight to the right document silo and parse metadata filters cleanly!

## 8. Video Chapter Outline
1. **When Single-Query Search Fails** — Multi-doc crowding & vocabulary mismatch.
2. **Multi-Query + RAG-Fusion RRF** — Plain-word explanation + hand-math proof.
3. **Query Decomposition Live Comparison** — Beating NB05 baseline on compound questions.
4. **Step-Back & HyDE** — Bridging abstract concepts and prose wording.
5. **Cost & When-Not-To-Use Matrix** — Choosing translation selectively.